In [143]:
import numpy as np 
import torch
import torch.nn as nn
import torch.optim as optim
punch_data = pd.read_csv("punch.csv",usecols=range(13))
punch_data = pd.DataFrame(punch_data)
no_punch_data = pd.read_csv("normal.csv",usecols=range(13))
no_punch_data = pd.DataFrame(normal_data)
# print(punch_data.shape)
# print(normal_data.shape)
data = np.concatenate([normal_data,punch_data],axis=0)
print(data.shape)
Y_data = np.concatenate([np.ones((158,1)),np.zeros((164,1))],axis=0)
Y_data.shape

(322, 13)


(322, 1)

In [1]:
# CSV 데이터 불러오기
import numpy as np
import pandas as pd
punch_data = pd.read_csv("punch.csv", usecols=range(13))
no_punch_data = pd.read_csv("normal.csv", usecols=range(13))
pet_data = pd.read_csv("pet.csv", usecols=range(13))
pinch_data = pd.read_csv("pinch.csv", usecols=range(13))
print(punch_data)

FileNotFoundError: [Errno 2] No such file or directory: 'punch.csv'

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

# 데이터 로드 및 변환
timesteps = 13
features = 1

# CSV 데이터 불러오기
punch_data = pd.read_csv("punch.csv", usecols=range(13))
no_punch_data = pd.read_csv("normal.csv", usecols=range(13))
pet_data = pd.read_csv("pet.csv", usecols=range(13))
pinch_data = pd.read_csv("pinch.csv", usecols=range(13))

# NumPy 변환 후 float32 형 변환
punch_data = np.array(punch_data).astype(np.float32)
no_punch_data = np.array(no_punch_data).astype(np.float32)
pet_data = np.array(pet_data).astype(np.float32)
pinch_data = np.array(pinch_data).astype(np.float32)

# 샘플 개수
samples_punch = len(punch_data)
samples_no_punch = len(no_punch_data)
samples_pet = len(pet_data)
samples_pinch = len(pinch_data)

# 🔹 데이터 합치기 (Pinch 추가)
X_data = np.concatenate([punch_data, no_punch_data, pet_data, pinch_data], axis=0)  # (총 샘플 개수, 13)
X_data = np.expand_dims(X_data, axis=-1)  # (총 샘플 개수, 13, 1)

# 🔹 레이블 생성 (0: 펀치, 1: 비펀치, 2: 애완동물, 3: 핀치)
y_data = np.concatenate([
    np.zeros((samples_punch, 1)),  # 0: 펀치
    np.ones((samples_no_punch, 1)),  # 1: 비펀치
    np.full((samples_pet, 1), 2),  # 2: 애완동물
    np.full((samples_pinch, 1), 3)  # 3: 핀치
], axis=0)

# PyTorch Tensor 변환
X_data = torch.tensor(X_data, dtype=torch.float32)
y_data = torch.tensor(y_data, dtype=torch.long).squeeze()

# 데이터셋 나누기
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

# DataLoader 사용
batch_size = 4
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

#  LSTM 모델 정의 (🔹 num_classes=4 로 변경)
class PunchDetectionLSTM(nn.Module):
    def __init__(self, num_classes=4):
        super(PunchDetectionLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, num_layers=2, batch_first=True)
        self.fc1 = nn.Linear(64, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, num_classes)  # 🔹 4개 클래스 (펀치, 비펀치, 애완동물, 핀치)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        x = self.fc1(lstm_out[:, -1, :])  # 마지막 타임스텝 사용
        x = self.relu(x)
        x = self.fc2(x)  # 🔹 Softmax 사용하지 않음 (CrossEntropyLoss 내에서 포함됨)
        return x

#  모델 학습
model = PunchDetectionLSTM()
criterion = nn.CrossEntropyLoss()  # 🔹 다중 분류를 위한 CrossEntropyLoss
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # 검증 데이터 평가
    model.eval()
    val_loss = 0.0
    correct, total = 0, 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()

            # 🔹 정확도 계산
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss / len(train_loader):.4f}, "
          f"Val Loss: {val_loss / len(test_loader):.4f}, Accuracy: {accuracy:.2f}%")

# 모델 저장
torch.save(model.state_dict(), "please.pth")
print(" 모델이 저장되었습니다!")


Epoch [1/20], Train Loss: 0.7000, Val Loss: 0.2769, Accuracy: 91.33%
Epoch [2/20], Train Loss: 0.3437, Val Loss: 0.2440, Accuracy: 90.00%
Epoch [3/20], Train Loss: 0.2623, Val Loss: 0.2820, Accuracy: 86.00%
Epoch [4/20], Train Loss: 0.2190, Val Loss: 0.2652, Accuracy: 91.33%
Epoch [5/20], Train Loss: 0.1979, Val Loss: 0.1927, Accuracy: 92.67%
Epoch [6/20], Train Loss: 0.1727, Val Loss: 0.1879, Accuracy: 92.67%
Epoch [7/20], Train Loss: 0.1427, Val Loss: 0.1467, Accuracy: 95.33%
Epoch [8/20], Train Loss: 0.1175, Val Loss: 0.1054, Accuracy: 96.67%
Epoch [9/20], Train Loss: 0.1516, Val Loss: 0.1342, Accuracy: 94.67%
Epoch [10/20], Train Loss: 0.0988, Val Loss: 0.0955, Accuracy: 97.33%
Epoch [11/20], Train Loss: 0.1299, Val Loss: 0.1280, Accuracy: 94.00%
Epoch [12/20], Train Loss: 0.1074, Val Loss: 0.2034, Accuracy: 92.67%
Epoch [13/20], Train Loss: 0.0867, Val Loss: 0.1256, Accuracy: 95.33%
Epoch [14/20], Train Loss: 0.1289, Val Loss: 0.1309, Accuracy: 94.67%
Epoch [15/20], Train Loss: 0.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import serial

class PunchDetectionLSTM(nn.Module):
    def __init__(self, num_classes=4):
        super(PunchDetectionLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, num_layers=2, batch_first=True)
        self.fc1 = nn.Linear(64, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, num_classes)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        x = self.fc1(lstm_out[:, -1, :])
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = PunchDetectionLSTM()
model.load_state_dict(torch.load("please.pth"))
model.eval()

SERIAL_PORT = "COM11"
BAUD_RATE = 9600

def processmodel(values):
    new_data = np.array(values).reshape(1, 13, 1)
    new_data_tensor = torch.tensor(new_data, dtype=torch.float32)

    with torch.no_grad():
        predicted_value = model(new_data_tensor)

    probabilities = torch.softmax(predicted_value, dim=1).numpy()
    probabilities = np.round(probabilities, 4)
    predicted_class = torch.argmax(predicted_value, dim=1).item()

    class_labels = {0: "펀치", 1: "노말", 2: "쓰다듬기", 3: "꼬집기"}

    print(f"예측 결과: {class_labels[predicted_class]} (확률: {probabilities[0][0]:.4f}, {probabilities[0][1]:.4f}, {probabilities[0][2]:.4f}, {probabilities[0][3]:.4f})\n")

try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    print(f"시리얼 포트 {SERIAL_PORT} 연결됨!")

    while True:
        line = ser.readline().decode('utf-8').strip().rstrip(",")

        if line:
            try:
                values = [float(x) for x in line.split(',')]

                if len(values) == 13:
                    processmodel(values)
                else:
                    print(f"데이터 길이 오류 (13개 필요, 현재: {len(values)}) → {values}")

            except ValueError:
                print(f"변환 오류: {line}")

except serial.SerialException as e:
    print(f"시리얼 포트 연결 실패: {e}")
except KeyboardInterrupt:
    print("프로그램 종료")
finally:
    if 'ser' in locals() and ser.is_open:
        ser.close()
        print(f"시리얼 포트 {SERIAL_PORT} 닫힘")


시리얼 포트 COM11 연결됨!
예측 결과: 노말 (확률: 0.0008, 0.9751, 0.0240, 0.0000)

예측 결과: 노말 (확률: 0.0008, 0.9751, 0.0240, 0.0000)

예측 결과: 노말 (확률: 0.0008, 0.9750, 0.0241, 0.0000)

예측 결과: 노말 (확률: 0.0008, 0.9749, 0.0242, 0.0000)

예측 결과: 노말 (확률: 0.1389, 0.7414, 0.1153, 0.0043)

예측 결과: 펀치 (확률: 0.9529, 0.0378, 0.0082, 0.0012)

예측 결과: 노말 (확률: 0.0008, 0.9751, 0.0240, 0.0000)

예측 결과: 노말 (확률: 0.0008, 0.9752, 0.0239, 0.0000)

예측 결과: 노말 (확률: 0.0008, 0.9752, 0.0240, 0.0000)

예측 결과: 쓰다듬기 (확률: 0.0000, 0.0000, 0.9962, 0.0038)

예측 결과: 노말 (확률: 0.0009, 0.9747, 0.0244, 0.0000)

예측 결과: 노말 (확률: 0.0008, 0.9752, 0.0240, 0.0000)

예측 결과: 노말 (확률: 0.2980, 0.5683, 0.1268, 0.0069)

예측 결과: 쓰다듬기 (확률: 0.0015, 0.0035, 0.9624, 0.0325)

예측 결과: 쓰다듬기 (확률: 0.0000, 0.0001, 0.9452, 0.0547)

예측 결과: 노말 (확률: 0.0019, 0.9681, 0.0300, 0.0001)

예측 결과: 노말 (확률: 0.0008, 0.9753, 0.0238, 0.0000)

예측 결과: 노말 (확률: 0.0008, 0.9752, 0.0240, 0.0000)

예측 결과: 펀치 (확률: 0.8296, 0.0273, 0.0551, 0.0880)

예측 결과: 노말 (확률: 0.0156, 0.9401, 0.0440, 0.0003)

예측 결과: 노말 (확률: 0

svm

In [2]:
import joblib
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# 데이터를 2D 형태로 변환 (샘플 수, 13) -> SVM은 2D 데이터 필요
# X_data_np = X_data.numpy().reshape(X_data.shape[0], -1)  # PyTorch Tensor → NumPy 변환
X_data_mean = np.mean(X_data.numpy().reshape(X_data.shape[0], -1), axis=1).reshape(-1, 1)  # (샘플 수, 1)

print(X_data_mean)
y_data_np = y_data.numpy()

# 데이터셋 나누기
X_train, X_test, y_train, y_test = train_test_split(X_data_mean, y_data_np, test_size=0.2, random_state=42)

# SVM 모델 생성 및 학습 (RBF 커널 사용)
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_model.fit(X_train, y_train)

# 예측 수행
y_pred = svm_model.predict(X_test)

# 정확도 평가
accuracy = accuracy_score(y_test, y_pred)
print(f"SVM Test Accuracy: {accuracy * 100:.2f}%")

# SVM 모델 저장
joblib.dump(svm_model, "svm_model.pkl")
print("SVM 모델이 'svm_model.pkl' 파일로 저장되었습니다!")


NameError: name 'np' is not defined

In [1]:
import numpy as np
import joblib
import serial

# 저장된 SVM 모델 로드
svm_model = joblib.load("svm_model.pkl")

# 시리얼 포트 설정
SERIAL_PORT = "COM11"
BAUD_RATE = 9600

# 클래스 라벨 정의
class_labels = {0: "펀치", 1: "노말", 2: "쓰다듬기", 3: "꼬집기"}

def process_svm_model(values):
    """
    SVM 모델을 사용하여 실시간 데이터 분류
    """
    new_data = np.array(values).reshape(1, -1)  # SVM은 2D 입력 필요

    # 예측 수행
    predicted_class = svm_model.predict(new_data)[0]

    print(f"예측 결과: {class_labels[predicted_class]}\n")

try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    print(f"시리얼 포트 {SERIAL_PORT} 연결됨!")

    while True:
        line = ser.readline().decode('utf-8').strip().rstrip(",")

        if line:
            try:
                values = [float(x) for x in line.split(',')]

                if len(values) == 13:
                    avg = np.mean(values)

                    process_svm_model(avg)
                    
                else:
                    print(f"데이터 길이 오류 (13개 필요, 현재: {len(values)}) → {values}")

            except ValueError:
                print(f"변환 오류: {line}")

except serial.SerialException as e:
    print(f"시리얼 포트 연결 실패: {e}")
except KeyboardInterrupt:
    print("프로그램 종료")
finally:
    if 'ser' in locals() and ser.is_open:
        ser.close()
        print(f"시리얼 포트 {SERIAL_PORT} 닫힘")


시리얼 포트 COM11 연결됨!
예측 결과: 꼬집기

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 펀치

예측 결과: 쓰다듬기

예측 결과: 노말

예측 결과: 쓰다듬기

예측 결과: 펀치

예측 결과: 노말

예측 결과: 노말

예측 결과: 노말

예측 결과: 펀치

예측 결과: 쓰다듬기

데이터 길이 오류 (13개 필요, 현재: 4) → [30.14, 28.96, 58.5, 31.94]
프로그램 종료
시리얼 포트 COM11 닫힘


In [3]:
print(X_data.mean())

NameError: name 'X_data' is not defined

In [2]:
X_data.describe()

NameError: name 'X_data' is not defined